<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12.1-power-grid-stability-estimation/Ex12.1_05_compare_and_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_12.1 · Notebook 05 — Compare, and Write It Up

**Paired with L12.1 · Power Grid Stability Estimation**

**Prerequisite: all previous notebooks.**

This notebook collects what you produced and turns it into the report. It
computes almost nothing new — its job is to make you put the numbers next to
each other, which is where the conclusions actually are.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12.1-power-grid-stability-estimation/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
ref = pb.load("00_reference")
wls = pb.load("01_wls")
alg = pb.load("02_algebraic")
dyn = pb.load("03_dynamic")
inr = pb.load("04_inertia")

V_true, th_true = ref["V"], ref["th"]
ms_thin = pb.thin_measurements()
metered = ms_thin.measured_buses()

r_wls = pb.error_table(V_true, th_true, wls["V_thin"], wls["th_thin"],
                       metered, label="WLS, thin set")
r_pinn = pb.error_table(V_true, th_true, alg["V_pinn"], alg["th_pinn"],
                        metered, label="PINN, thin set")
pb.comparison_table([("WLS (thin)", r_wls), ("PINN (thin)", r_pinn)])

**Expected output**

> The PINN's unmetered-bus error should be clearly below WLS's. If it is not,
> say so — an honest negative, investigated, is worth more than a tuned positive,
> and the first thing to check is your λ.

## The five relations that must hold

If any of these comes out backwards, something is wrong with the
implementation rather than interesting about the physics. This is the fastest
debugging tool in the exercise.

In [ ]:
checks = [
    ("full metering: WLS is accurate",            None),
    ("thin metering: WLS degrades badly",         None),
    ("thin metering: PINN beats WLS unmetered",   None),
    ("quiet window: inertia estimate unreliable", None),
    ("wrong topology: small residual, wrong state", None),
]
# TODO: replace each None with True/False computed from your own results,
#       and print the table. Do not hand-wave: each one is checkable.
for name, val in checks:
    print(f"  [{'?' if val is None else ('OK' if val else 'FAIL')}]  {name}")

## TODO — the report

Fill in the sections below and generate the document. The final question is the
one that carries the marks.

In [ ]:
sections = [
    ("Measurement sets and observability",
     "TODO: state both sets, their ranks, and which buses were unmetered."),
    ("WLS baseline",
     "TODO: your numbers from notebook 01, both measurement sets."),
    ("The lambda sweep",
     "TODO: the curve, the value you chose, and why. State what happens at "
     "both extremes."),
    ("Estimation error, split",
     "TODO: metered vs unmetered, worst and mean. Not the mean alone."),
    ("Inertia",
     "TODO: your H for machine 1, the window you used, and how much you trust "
     "it. Include the quiet-window result."),
    ("RoCoF and its window",
     "TODO: your three window values and what you concluded."),
    ("What depends on the assumed impedances",
     "TODO: the network is DK2-representative. Which of your conclusions "
     "would survive if the reactances were wrong by 20 per cent?"),
]
pb.make_report(sections,
               os.path.join(pb.RESULTS, "Ex12.1_report.md"),
               author="TODO: your name")

**Expected output**

> `wrote Ex12.1_outputs/Ex12.1_report.md`
>
> Open it, fill in every TODO, and read the last section twice before you submit.

---

## Before you hand it in

1. Every table separates metered from unmetered buses, and reports the worst
   bus as well as the mean. If one does not, that table is misleading and you
   know it.
2. Every number carries the seed and the measurement set it came from.
3. The λ you chose is justified in a sentence, not asserted.
4. The inertia you report is machine 1's, and the reason you are not reporting
   machine 0's is written down.
5. You have said which of your conclusions depend on the assumed line
   impedances, and which depend only on the structure of the network.